In [6]:
# Download packages
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    mean_squared_error,
    mean_absolute_error,
    r2_score
)
import pandas as pd
import numpy as np
from pathlib import Path


In [19]:
# Download dataset
regression_dataset_1 = pd.read_csv("../data/processed/regression_dataset_2.csv")
regression_dataset_1["abs_congestion"] = np.abs(regression_dataset_1["congestion"])

In [21]:
# categorical hour
regression_dataset_1["hour_type"] = (
    pd.to_datetime(regression_dataset_1["hour_ending"]).dt.hour
)

# optional log target
regression_dataset_1["log_congestion"] = np.log1p(
    regression_dataset_1["abs_congestion"]
)

features = [
    "EC_load_weighted",
    "BC_load_weighted",
    "capacity_tightness",
    "temperature",
    "region",
    "hour_type"
]

X = regression_dataset_1[features]
y = regression_dataset_1["log_congestion"]

target = "log_congestion"

model_data = regression_dataset_1[features + [target]].dropna()

X = model_data[features]
y = model_data[target]

y


0        0.000000
1        0.000000
2        0.000000
3        0.000000
4        0.000000
           ...   
43670    0.328104
43671    0.308709
43672    0.314202
43673    0.301955
43674    0.747083
Name: log_congestion, Length: 43670, dtype: float64

In [22]:
# One hot encoding
X = pd.get_dummies(
    X,
    columns=["region", "hour_type"],
    drop_first=True
)

In [23]:
# Train-test split
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

In [24]:
# Training model
rf = RandomForestRegressor(
    n_estimators=200,
    max_depth=10,
    random_state=42,
    n_jobs=-1
)

rf.fit(X_train, y_train)

RandomForestRegressor(max_depth=10, n_estimators=200, n_jobs=-1,
                      random_state=42)

In [25]:
# Check the amoun t of losing data
print("Before:", regression_dataset_1.shape)
print("After:", model_data.shape)
print(regression_dataset_1[features + [target]].isna().sum())

Before: (43675, 25)
After: (43670, 7)
EC_load_weighted      0
BC_load_weighted      0
capacity_tightness    0
temperature           0
region                0
hour_type             0
log_congestion        5
dtype: int64


In [26]:
# Prediction
y_pred = rf.predict(X_test)

In [27]:
# Evaluation
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
mae = mean_absolute_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)

print("RMSE:", rmse)
print("MAE:", mae)
print("R²:", r2)

RMSE: 0.9314299144014153
MAE: 0.7640315024841084
R²: 0.15411097688461495


In [ ]:
# Check at the end
importance = pd.DataFrame({
    "feature": X.columns,
    "importance": rf.feature_importances_
})

importance = importance.sort_values(
    by="importance",
    ascending=False
)

print(importance.head(20))

               feature  importance
2   capacity_tightness    0.309999
3          temperature    0.224724
1     BC_load_weighted    0.162055
0     EC_load_weighted    0.126535
15         hour_type_8    0.038162
16         hour_type_9    0.027657
14         hour_type_7    0.017586
17        hour_type_10    0.016016
4          region_EAST    0.009576
13         hour_type_6    0.009560
18        hour_type_11    0.008366
22        hour_type_15    0.006341
23        hour_type_16    0.005221
19        hour_type_12    0.004407
12         hour_type_5    0.003722
21        hour_type_14    0.003574
20        hour_type_13    0.003473
11         hour_type_4    0.003020
6         region_SOUTH    0.002729
25        hour_type_18    0.002621


In [29]:
# Now try hyper parameter tuning
from sklearn.model_selection import GridSearchCV
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import numpy as np

rf = RandomForestRegressor(random_state=42, n_jobs=-1)

param_grid = {
    "n_estimators": [100, 200, 300],
    "max_depth": [5, 10, 15, None],
    "min_samples_split": [2, 5, 10],
    "min_samples_leaf": [1, 2, 5],
    "max_features": ["sqrt", "log2", None]
}

grid = GridSearchCV(
    estimator=rf,
    param_grid=param_grid,
    scoring="r2",
    cv=5,
    n_jobs=-1,
    verbose=1
)

grid.fit(X_train, y_train)

print("Best parameters:", grid.best_params_)
print("Best CV R²:", grid.best_score_)

Fitting 5 folds for each of 324 candidates, totalling 1620 fits


KeyboardInterrupt: 